In [1]:
# This lesson uses the "dogs" dataset
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

dogs = pd.read_csv("datasets/ShelterDogs.csv")

# Setting category variables


In [2]:
dogs.info()

<class 'pandas.DataFrame'>
RangeIndex: 2937 entries, 0 to 2936
Data columns (total 19 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   ID                 2937 non-null   int64  
 1   name               2845 non-null   str    
 2   age                2937 non-null   float64
 3   sex                2937 non-null   str    
 4   breed              2937 non-null   str    
 5   date_found         2937 non-null   str    
 6   adoptable_from     2937 non-null   str    
 7   posted             2937 non-null   str    
 8   color              2937 non-null   str    
 9   coat               2937 non-null   str    
 10  size               2937 non-null   str    
 11  neutered           1852 non-null   str    
 12  housebroken        460 non-null    str    
 13  likes_people       1999 non-null   str    
 14  likes_children     1219 non-null   str    
 15  get_along_males    1633 non-null   str    
 16  get_along_females  1673 non-null   

Thoughts: 
- Some of these have obvious choices for what to fill the null values with. Maybe play around with the data? 
- Data Questions:
    - Is there a relationship between breed and whether they like people/children?
    - Assuming "get_along_males" and "get_along_females" refers to other dogs, are certain dogs generally sociable? 

## Converting coat variable to a category
- Using `.astype("category")`

In [7]:
print("Memory usage as a string series: ", dogs["coat"].nbytes)
dogs["coat"] = dogs["coat"].astype("category")
print("Memory as a category series: ", dogs["coat"].nbytes)
print("\nFrequency Table")
print(dogs["coat"].value_counts(dropna=False))

Memory usage as a string series:  2969
Memory as a category series:  2969

Frequency Table
coat
short         1972
medium         565
wirehaired     220
long           180
Name: count, dtype: int64


- Luke note: To convert a variable to ordered category, cast with `.astype('category')` then use `df['col'].reorder_categories([order], ordered=True)`, or the .cat.set_categories() approach from later in the notebook
- Read up later: Why the heck does a categorical series use so much less memory?! Some kind of pointer based system? (ie. Only need memory for known categories rather than every possible string?)

## The .cat accessor object
- Allows direct access and manipulation of a categorical Series  

 `Series.cat.method_name`  
Common parameters:
- `new_categories`: a list of new categories to add
- `inplace`: a bool for overwriting current series
- `ordered`: Specify ordinal vs. nominal

## Setting Series categories

In [8]:
dogs["coat"] = dogs["coat"].cat.set_categories(
    new_categories=["short", "medium", "long"]
)
dogs["coat"].value_counts(dropna=False)


coat
short     1972
medium     565
NaN        220
long       180
Name: count, dtype: int64

- By changing the categories in the series, all the "wirehaired" entries are replace with NaN.

## Setting order
- add `ordered=True` kwarg to `df['col'].cat.set_categories()`

In [9]:
dogs["coat"] = dogs["coat"].cat.set_categories(
    new_categories=["short", "medium", "long"],
    ordered=True
)
dogs["coat"].head(3)

0    short
1    short
2    short
Name: coat, dtype: category
Categories (3, str): ['short' < 'medium' < 'long']

## Adding/Removing Categories

In [11]:
dogs["likes_people"].value_counts(dropna=False)

likes_people
yes    1991
NaN     938
no        8
Name: count, dtype: int64

- Almost 1/3rd of rows are null!
- Meaning of NaN could vary. Didn't check vs. Couldn't tell?
- Add categories using `df['col'].cat.add_categories()`

In [14]:
dogs["likes_people"] = dogs["likes_people"].astype("category")
dogs["likes_people"] = dogs["likes_people"].cat.add_categories(
    new_categories=["did not check", "could not tell"]
)
dogs["likes_people"].cat.categories

Index(['no', 'yes', 'did not check', 'could not tell'], dtype='str')

- Remove categories using `df['col'].cat.remove_categories(removals=[list])`

# Updating categories

## Renaming categories
- use `Series.cat.rename_categories(new_categories=dict)`
    - Dict is structured as {"current category": "new category"}

In [17]:
dogs["breed"] = dogs["breed"].astype("category")
dogs["breed"] = dogs["breed"].cat.rename_categories(new_categories = {"Unknown Mix": "Unknown"})
dogs["breed"].value_counts()

breed
Unknown                            1524
German Shepherd Dog Mix             190
Dachshund Mix                       147
Labrador Retriever Mix               83
Staffordshire Terrier Mix            62
                                   ... 
Tibetan Terrier                       1
Tibetan Terrier Mix                   1
Vizsla, Whippet Mix                   1
West Highland White Terrier Mix       1
Yorkshire Terrier                     1
Name: count, Length: 277, dtype: int64

- Can rename categories using a lambda function (also note `str.title()` puts stuff into title case)

In [24]:
dogs['breed'] = dogs['breed'].cat.rename_categories(lambda c: (print(c), c.title()))

Adoptable From:
Afghan Hound
Akita
Akita Mix
Akita, Boxer Mix
Akita, German Shepherd Dog Mix
Akita, Labrador Retriever Mix
Alaskan Malamute, Caucasian Ovtcharka, German Shepherd Dog Mix
American Bulldog
American Bulldog Mix
American Bulldog, Dogo Argentino Mix
American Pit Bull Terrier, Staffordshire Terrier Mix
Australian Cattle Dog, German Shepherd Dog Mix
Australian Shepherd, Labrador Retriever Mix
Azawakh, Hungarian Greyhound Mix
Basset Hound Mix
Basset Hound, Dachshund Mix
Beagle
Beagle Mix
Beagle, Chihuahua Mix
Beagle, Dachshund Mix
Beagle, Hound Mix
Beagle, Labrador Retriever, Scotch Collie Mix
Bearded Collie Mix
Bearded Collie, Labrador Retriever Mix
Bearded Collie, Staffordshire Terrier Mix
Belgian Malinois
Belgian Malinois Mix
Belgian Malinois, Belgian Sheepdog Mix
Belgian Malinois, German Shepherd Dog Mix
Belgian Malinois, Greyhound Mix
Belgian Malinois, Hungarian Vizsla Mix
Belgian Malinois, Labrador Retriever Mix
Belgian Sheepdog Mix
Belgian Sheepdog, German Shepherd Dog M

## Collapsing categories
- Use `df['col'].replace()`
- For example, turning extra detail into broad classifications
- Note, need to use categories that were already specified!
- Also, the new category created will not *stay* categorical

In [30]:
data = [["cleric"], ["fighter"], ["fighter"], ["wizard"], ["rogue"], ["sorcerer"]]
df = pd.DataFrame(data, columns=["class"])
df['class'] = df['class'].astype('category')
update_classes = {
    "fighter": "fighter",
    "rogue": "fighter",
    "cleric": "wizard",
    "wizard": "wizard",
}
df['class-type'] = df['class'].replace(update_classes)

# Reordering categories

- Can use `.cat.reorder_categories(new_categories = list, ordered=True, inplace=True)`
- Worth noting `df.groupby` will return data in the specified order *even if the categorical variable is not ordered!*

# Cleaning and accessing data

## Possible issues with categorical data
1) Inconsistent values (ex. `"Ham"`, `"ham"`, `" Ham"`)
2) Misspelled values
3) Wrong datatype

## Identifying issues
Use `series.cat.categories` or `series.value_counts()`

In [31]:
dogs["get_along_cats"].value_counts()

get_along_cats
yes    275
no     156
Name: count, dtype: int64

## Fixing issues
- Whitespace -> `df['col'].str.strip()`
- Capitalization -> `df['col'].str.title()`, `.upper()`, `.lower()`
- Misspelled words -> Create a "replace_map" dictionary, use `df['col'].replace(replace_map, inplace=True)`

## Using the str accessor object
- Search for a substring within each entry: ex. `dogs['breed'].str.contains('Shepherd',regex=False)`

## Accessing data with loc
- Can access series values based on category

In [64]:
import time

durations = []
for i in range(10000):
    start_time = time.perf_counter()
    likes_cats = dogs.loc[dogs["get_along_cats"]=="yes","size"]
    end_time = time.perf_counter()
    durations.append(end_time-start_time)

print(f"Average Execution Time: {np.mean(durations):.5f} seconds")

Average Execution Time: 0.00026 seconds


In [65]:
durations = []
for i in range(10000):
    start_time = time.perf_counter()
    likes_cats = dogs[dogs["get_along_cats"]=='yes']['size']
    end_time = time.perf_counter()
    durations.append(end_time-start_time)

print(f"Execution time: {end_time - start_time:.5f} seconds")

Execution time: 0.00044 seconds


### TIL: Why .loc[] and not chained indexing 
- I was wondering about the point of using .loc[] when you can get the same result using chained indexing (as seen in the two cells above). When *reading* they're equivalent, but behind the scenes it matters that you're doing multiple steps. `dogs[dogs["get_along_cats"]=='yes']['size']` means that Python first returns all the columns the dataset (filtered to certain rows) *then* subsets to the size column, but .loc[] goes straight to the desired entries
- Use .loc[] when trying to write new values, because .loc[] means you're actually working with the original df, rather than a throwaway copy. 

In [68]:
(0.00044-0.00026)/0.00026

0.6923076923076925